# Finite-Size Scaling Analysis â€” 3D Ising Model

This notebook performs finite-size scaling (FSS) analysis on Monte Carlo sweep data
generated by the `fss` Rust binary. It extracts the critical temperature Tc and
critical exponents Î², Î³, Î±, Î½ by analysing how observables scale with lattice size N.

**Generate data first:**
```bash
cargo run --release --bin fss -- --sizes 8,12,16,20,24,28 --wolff --outdir analysis/data
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
from scipy.optimize import minimize_scalar, differential_evolution
from scipy.interpolate import interp1d, UnivariateSpline
from pathlib import Path
from pub_style import setup_style, COLORS as PUB_COLORS, label_panel, jackknife_error

setup_style()

DATA_DIR = Path('data')

# 3D Ising universality class â€” theory values
NU    = 0.6301
BETA  = 0.3265
ALPHA = 0.1096
GAMMA = 1.2372
TC_THEORY = 4.5115

N_BLOCKS = 20  # jackknife blocks for error estimation

def compute_observables_from_raw(raw_df, n):
    """Compute observables + jackknife errors from raw sample data for each temperature."""
    V = n**3
    temps = sorted(raw_df['T'].unique())
    rows = []
    
    for T in temps:
        beta = 1.0 / T
        mask = np.abs(raw_df['T'] - T) < 1e-5
        sub = raw_df[mask]
        e = sub['e'].values        # energy per spin
        m = sub['m_abs'].values    # |m| per spin
        ms = sub['m_signed'].values  # signed m per spin
        
        n_samples = len(e)
        block_size = n_samples // N_BLOCKS
        trimmed = N_BLOCKS * block_size
        e, m, ms = e[:trimmed], m[:trimmed], ms[:trimmed]
        
        # Full estimates
        E_avg = np.mean(e)
        M_avg = np.mean(m)
        M2_avg = np.mean(m**2)
        M4_avg = np.mean(m**4)
        Cv_full = beta**2 * V * np.var(e, ddof=0)
        chi_conn_full = beta * V * (M2_avg - M_avg**2)
        U_full = 1.0 - M4_avg / (3.0 * M2_avg**2) if M2_avg > 1e-15 else 0.0
        
        # Jackknife
        jk_E = np.zeros(N_BLOCKS)
        jk_M = np.zeros(N_BLOCKS)
        jk_Cv = np.zeros(N_BLOCKS)
        jk_chi = np.zeros(N_BLOCKS)
        jk_U = np.zeros(N_BLOCKS)
        
        for b in range(N_BLOCKS):
            idx = np.concatenate([np.arange(0, b*block_size),
                                  np.arange((b+1)*block_size, trimmed)])
            eb, mb = e[idx], m[idx]
            
            jk_E[b] = np.mean(eb)
            jk_M[b] = np.mean(mb)
            m2 = np.mean(mb**2)
            m4 = np.mean(mb**4)
            jk_Cv[b] = beta**2 * V * np.var(eb, ddof=0)
            jk_chi[b] = beta * V * (m2 - np.mean(mb)**2)
            jk_U[b] = 1.0 - m4 / (3.0 * m2**2) if m2 > 1e-15 else 0.0
        
        jk_f = (N_BLOCKS - 1) / N_BLOCKS
        rows.append({
            'T': T,
            'E': E_avg, 'M': M_avg, 'M2': M2_avg, 'M4': M4_avg,
            'Cv': Cv_full, 'chi_conn': chi_conn_full, 'U': U_full,
            'E_err': np.sqrt(jk_f * np.sum((jk_E - E_avg)**2)),
            'M_err': np.sqrt(jk_f * np.sum((jk_M - M_avg)**2)),
            'Cv_err': np.sqrt(jk_f * np.sum((jk_Cv - Cv_full)**2)),
            'chi_err': np.sqrt(jk_f * np.sum((jk_chi - chi_conn_full)**2)),
            'U_err': np.sqrt(jk_f * np.sum((jk_U - U_full)**2)),
        })
    
    return pd.DataFrame(rows)

# Load raw data and compute observables with errors
dfs = {}
for p in sorted(DATA_DIR.glob('fss_raw_N*.csv')):
    n = int(p.stem.split('N')[1])
    print(f'Loading & processing N={n}...', end=' ', flush=True)
    raw = pd.read_csv(p)
    df = compute_observables_from_raw(raw, n)
    dfs[n] = df
    print(f'{len(df)} temps, T=[{df["T"].min():.2f}, {df["T"].max():.2f}], '
          f'chi_max={df["chi_conn"].max():.1f}Â±{df.loc[df["chi_conn"].idxmax(), "chi_err"]:.1f}')

if not dfs:
    raise SystemExit('No raw data found. Run: cargo run --release --bin fss -- --wolff --raw ...')

SIZES = sorted(dfs.keys())
COLORS = cm.viridis(np.linspace(0.15, 0.85, len(SIZES)))
print(f'\nLoaded sizes: {SIZES}')

dT = dfs[SIZES[0]]['T'].diff().median()
print(f'Temperature spacing: dT = {dT:.4f}')

## 1. Raw observables for all N

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 5.5))
labels = ['$\\langle e \\rangle / J$', '$|m|$', '$C_v$', '$\\chi_{\\mathrm{conn}}$']
cols   = ['E', 'M', 'Cv', 'chi_conn']
errs   = ['E_err', 'M_err', 'Cv_err', 'chi_err']
panels = ['a', 'b', 'c', 'd']

for ax, col, err_col, lab, pan in zip(axes.flat, cols, errs, labels, panels):
    for (n, df), c in zip(dfs.items(), COLORS):
        ax.plot(df['T'], df[col], '-', color=c, lw=0.8, label=f'$L={n}$')
        ax.fill_between(df['T'], df[col] - df[err_col], df[col] + df[err_col],
                        color=c, alpha=0.12)
    ax.set_xlabel('$T \\; [J/k_B]$')
    ax.set_ylabel(lab)
    ax.axvline(TC_THEORY, color='red', lw=0.6, ls='--')
    label_panel(ax, pan)

axes[0,0].legend(fontsize=5, ncol=2, loc='lower left')
plt.tight_layout()
plt.savefig('fss_observables.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Binder Cumulant â€” Precise Tc

The Binder cumulant U = 1 - <M^4> / (3 <M^2>^2) crosses at the same Tc
for all N, giving a precise, finite-size-independent estimate of Tc.

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 3))

# Only plot T <= 4.8 to avoid high-T noise in Binder cumulant
for (n, df), c in zip(dfs.items(), COLORS):
    mask = df['T'] <= 4.8
    ax.plot(df.loc[mask, 'T'], df.loc[mask, 'U'], color=c, lw=0.8, label=f'$L={n}$')
    ax.fill_between(df.loc[mask, 'T'],
                    df.loc[mask, 'U'] - df.loc[mask, 'U_err'],
                    df.loc[mask, 'U'] + df.loc[mask, 'U_err'],
                    color=c, alpha=0.12)

ax.axvline(TC_THEORY, color='red', lw=0.6, ls='--')
ax.set_xlabel('$T \\; [J/k_B]$')
ax.set_ylabel('Binder cumulant $U$')
ax.legend(fontsize=5, ncol=2)
plt.tight_layout()
plt.savefig('fss_binder.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Load HIGH-RESOLUTION raw data and compute observables for Tc extraction ---
HIRES_DIR = DATA_DIR / 'hires'
print('\n=== Loading high-resolution raw data ===')

hires_dfs = {}
for p in sorted(HIRES_DIR.glob('fss_raw_N*.csv')):
    n = int(p.stem.split('N')[1])
    print(f'Loading hires N={n}...', end=' ', flush=True)
    raw = pd.read_csv(p)
    hires_dfs[n] = compute_observables_from_raw(raw, n)
    print(f'{len(hires_dfs[n])} temps')

hires_sizes = sorted(hires_dfs.keys())

# Extract Tc from HIRES Binder crossings (much cleaner than coarse data)
crossings = []
for i in range(len(hires_sizes)-1):
    n1, n2 = hires_sizes[i], hires_sizes[i+1]
    df1, df2 = hires_dfs[n1], hires_dfs[n2]
    T_common = np.linspace(
        max(df1['T'].min(), df2['T'].min()),
        min(df1['T'].max(), df2['T'].max()), 2000)
    f1 = interp1d(df1['T'], df1['U'], kind='cubic')
    f2 = interp1d(df2['T'], df2['U'], kind='cubic')
    diff = f1(T_common) - f2(T_common)
    for j in range(len(diff)-1):
        if diff[j] * diff[j+1] < 0:
            tc = T_common[j] - diff[j]*(T_common[j+1]-T_common[j])/(diff[j+1]-diff[j])
            crossings.append(tc)
            print(f'  N={n1}/{n2}: Tc = {tc:.4f}')
            break

if len(crossings) >= 2:
    # Use only the largest-L crossings (last 3) for best estimate
    best_crossings = crossings[-min(3, len(crossings)):]
    TC_BINDER = np.mean(best_crossings)
    TC_BINDER_ERR = np.std(best_crossings)
else:
    TC_BINDER = TC_THEORY
    TC_BINDER_ERR = 0.0

print(f'\nTc (Binder, hires) = {TC_BINDER:.4f} +/- {TC_BINDER_ERR:.4f}  (theory = {TC_THEORY})')
print(f'Error = {abs(TC_BINDER - TC_THEORY)/TC_THEORY*100:.2f}%')

# --- Histogram reweighting on fine T grid ---
print('\n=== Histogram reweighting ===')
T_fine = np.linspace(4.30, 4.70, 200)

rw_dfs = {}
rw_errs = {}

for p in sorted(HIRES_DIR.glob('fss_raw_N*.csv')):
    n = int(p.stem.split('N')[1])
    print(f'Reweighting N={n}...', end=' ', flush=True)
    raw_df = pd.read_csv(p)
    V = n**3
    sim_temps = sorted(raw_df['T'].unique())
    
    results = []
    err_results = []
    for T in T_fine:
        beta = 1.0 / T
        best_T = min(sim_temps, key=lambda t: abs(t - T))
        beta0 = 1.0 / best_T
        
        mask = np.abs(raw_df['T'] - best_T) < 1e-5
        sub = raw_df[mask]
        e_total = sub['e'].values * V
        m_abs = sub['m_abs'].values
        
        delta = beta - beta0
        log_w = -delta * e_total
        log_w -= log_w.max()
        w = np.exp(log_w)
        W = w.sum()
        
        avg_m = np.sum(w * m_abs) / W
        avg_m2 = np.sum(w * m_abs**2) / W
        avg_m4 = np.sum(w * m_abs**4) / W
        avg_e = np.sum(w * e_total) / W
        avg_e2 = np.sum(w * e_total**2) / W
        
        chi_conn = beta * V * (avg_m2 - avg_m**2)
        cv = beta**2 * (avg_e2 - avg_e**2) / V
        U = 1.0 - avg_m4 / (3.0 * avg_m2**2) if avg_m2 > 0 else 0.0
        
        results.append({'T': T, 'M': avg_m, 'M2': avg_m2, 'M4': avg_m4,
                       'chi_conn': chi_conn, 'Cv': cv, 'U': U, 'E': avg_e / V})
        
        # Jackknife
        n_samples = len(e_total)
        block_size = n_samples // N_BLOCKS
        jk_chi = np.zeros(N_BLOCKS)
        jk_cv = np.zeros(N_BLOCKS)
        jk_m = np.zeros(N_BLOCKS)
        jk_U = np.zeros(N_BLOCKS)
        
        for b in range(N_BLOCKS):
            idx = np.concatenate([np.arange(0, b*block_size),
                                  np.arange((b+1)*block_size, N_BLOCKS*block_size)])
            e_jk, m_jk = e_total[idx], m_abs[idx]
            log_wj = -delta * e_jk
            log_wj -= log_wj.max()
            wj = np.exp(log_wj)
            Wj = wj.sum()
            
            jm = np.sum(wj * m_jk) / Wj
            jm2 = np.sum(wj * m_jk**2) / Wj
            jm4 = np.sum(wj * m_jk**4) / Wj
            je = np.sum(wj * e_jk) / Wj
            je2 = np.sum(wj * e_jk**2) / Wj
            
            jk_chi[b] = beta * V * (jm2 - jm**2)
            jk_cv[b] = beta**2 * (je2 - je**2) / V
            jk_m[b] = jm
            jk_U[b] = 1.0 - jm4 / (3.0 * jm2**2) if jm2 > 0 else 0.0
        
        jk_f = (N_BLOCKS - 1) / N_BLOCKS
        err_results.append({
            'T': T,
            'chi_err': np.sqrt(jk_f * np.sum((jk_chi - chi_conn)**2)),
            'cv_err': np.sqrt(jk_f * np.sum((jk_cv - cv)**2)),
            'm_err': np.sqrt(jk_f * np.sum((jk_m - avg_m)**2)),
            'U_err': np.sqrt(jk_f * np.sum((jk_U - U)**2)),
        })
    
    rw_dfs[n] = pd.DataFrame(results)
    rw_errs[n] = pd.DataFrame(err_results)
    chi_max = rw_dfs[n]['chi_conn'].max()
    t_peak = rw_dfs[n].loc[rw_dfs[n]['chi_conn'].idxmax(), 'T']
    print(f'chi_peak={chi_max:.1f} at T={t_peak:.4f}')

print(f'Reweighted sizes: {sorted(rw_dfs.keys())}')

## 3. Peak Scaling & Exponent Extraction

Multiple methods for extracting critical exponents:
- **Peak scaling**: Ï‡_max ~ N^(Î³/Î½), restricted to well-resolved sizes
- **M(Tc)**: M(Tc) ~ N^(-Î²/Î½) 
- **Binder slope**: |dU/dT|_Tc ~ N^(1/Î½)
- **Hyperscaling check**: 2Î²/Î½ + Î³/Î½ = d = 3

In [ ]:
# --- Peak scaling from REWEIGHTED data (resolves true peak) ---
rw_sz = sorted(rw_dfs.keys())
logN = np.log(rw_sz)

# gamma/nu from chi peak
chi_peaks = []
chi_peak_errs = []
for n in rw_sz:
    idx = rw_dfs[n]['chi_conn'].idxmax()
    chi_peaks.append(rw_dfs[n].loc[idx, 'chi_conn'])
    chi_peak_errs.append(rw_errs[n].loc[idx, 'chi_err'])

slope_chi, _ = np.polyfit(logN, np.log(chi_peaks), 1)
gamma_nu_peak = slope_chi

# beta/nu from M(Tc)
m_at_tc = []
m_at_tc_errs = []
for n in rw_sz:
    rw = rw_dfs[n]
    f_m = interp1d(rw['T'], rw['M'], kind='cubic')
    f_m_err = interp1d(rw_errs[n]['T'], rw_errs[n]['m_err'], kind='linear')
    m_at_tc.append(float(f_m(TC_BINDER)))
    m_at_tc_errs.append(float(f_m_err(TC_BINDER)))

slope_m, _ = np.polyfit(logN, np.log(m_at_tc), 1)
beta_nu_meas = -slope_m

# 1/nu from Binder slope at Tc (use reweighted for all sizes)
du_at_tc = []
du_sizes = []
for n in rw_sz:
    rw = rw_dfs[n]
    f_U = interp1d(rw['T'], rw['U'], kind='cubic')
    dT_fine = 0.001
    dU = (f_U(TC_BINDER + dT_fine) - f_U(TC_BINDER - dT_fine)) / (2*dT_fine)
    du_at_tc.append(abs(dU))
    du_sizes.append(n)

slope_du, _ = np.polyfit(np.log(du_sizes), np.log(du_at_tc), 1)
nu_binder = 1.0 / slope_du

hyperscaling = 2*beta_nu_meas + gamma_nu_peak

print(f'gamma/nu = {gamma_nu_peak:.4f}  (theory = {GAMMA/NU:.4f}, error = {abs(gamma_nu_peak-GAMMA/NU)/(GAMMA/NU)*100:.1f}%)')
print(f'beta/nu  = {beta_nu_meas:.4f}  (theory = {BETA/NU:.4f}, error = {abs(beta_nu_meas-BETA/NU)/(BETA/NU)*100:.1f}%)')
print(f'nu       = {nu_binder:.4f}  (theory = {NU:.4f}, error = {abs(nu_binder-NU)/NU*100:.1f}%)')
print(f'2b/n + g/n = {hyperscaling:.4f}  (should = 3, error = {abs(hyperscaling-3)/3*100:.1f}%)')

# --- Publication plots with error bars ---
fig, axes = plt.subplots(1, 3, figsize=(7, 2.5))
Nfit = np.linspace(min(rw_sz)*0.9, max(rw_sz)*1.1, 100)

ax = axes[0]
ax.errorbar(rw_sz, chi_peaks, yerr=chi_peak_errs,
            fmt='o', ms=5, color=PUB_COLORS[0], capsize=3, zorder=5)
c0 = np.log(chi_peaks[0]) - slope_chi*np.log(rw_sz[0])
ax.plot(Nfit, np.exp(slope_chi*np.log(Nfit)+c0), '--', color=PUB_COLORS[1], lw=0.8,
        label=f'$\\gamma/\\nu={gamma_nu_peak:.3f}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$L$'); ax.set_ylabel('$\\chi_{\\mathrm{max}}$')
ax.legend(fontsize=6); label_panel(ax, 'a')

ax = axes[1]
ax.errorbar(rw_sz, m_at_tc, yerr=m_at_tc_errs,
            fmt='o', ms=5, color=PUB_COLORS[0], capsize=3, zorder=5)
c0m = np.log(m_at_tc[0]) - slope_m*np.log(rw_sz[0])
ax.plot(Nfit, np.exp(slope_m*np.log(Nfit)+c0m), '--', color=PUB_COLORS[1], lw=0.8,
        label=f'$\\beta/\\nu={beta_nu_meas:.3f}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$L$'); ax.set_ylabel('$M(T_c)$')
ax.legend(fontsize=6); label_panel(ax, 'b')

ax = axes[2]
ax.errorbar(du_sizes, du_at_tc, fmt='o', ms=5, color=PUB_COLORS[0], capsize=3, zorder=5)
Nfit2 = np.linspace(min(du_sizes)*0.9, max(du_sizes)*1.1, 100)
c0d = np.log(du_at_tc[0]) - slope_du*np.log(du_sizes[0])
ax.plot(Nfit2, np.exp(slope_du*np.log(Nfit2)+c0d), '--', color=PUB_COLORS[1], lw=0.8,
        label=f'$1/\\nu={slope_du:.3f}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$L$'); ax.set_ylabel('$|dU/dT|_{T_c}$')
ax.legend(fontsize=6); label_panel(ax, 'c')

plt.tight_layout()
plt.savefig('fss_peak_scaling.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Scaling Collapse â€” Ï‡(connected)

Plot N^(-Î³/Î½) Â· Ï‡_conn vs (T - Tc) Â· N^(1/Î½).
All curves should collapse onto one universal function when Î½ is correct.
We fix Tc from Binder crossing and optimise Î½.

In [ ]:
# Scaling collapse using REWEIGHTED fine-grid data
def collapse_quality_rw(nu, tc=TC_BINDER, exp_ratio=gamma_nu_peak):
    """Adjacent-point cost for chi_conn collapse."""
    all_x, all_y = [], []
    for n in rw_sz:
        rw = rw_dfs[n]
        x = (rw['T'] - tc) * n**(1/nu)
        y = rw['chi_conn'] / n**exp_ratio
        mask = (x > -15) & (x < 15)
        all_x.extend(x[mask].values)
        all_y.extend(y[mask].values)
    order = np.argsort(all_x)
    all_y_sorted = np.array(all_y)[order]
    rng = all_y_sorted.max() - all_y_sorted.min()
    return float(np.sum(np.diff(all_y_sorted)**2)) / rng**2

result = minimize_scalar(collapse_quality_rw, bounds=(0.4, 0.9), method='bounded')
nu_collapse = result.x
print(f'Optimised nu (chi collapse) = {nu_collapse:.4f}  (theory = {NU:.4f}, error = {abs(nu_collapse-NU)/NU*100:.1f}%)')

colors_rw = cm.viridis(np.linspace(0.15, 0.85, len(rw_sz)))
fig, ax = plt.subplots(figsize=(4.5, 3.5))
for i, n in enumerate(rw_sz):
    rw = rw_dfs[n]
    err = rw_errs[n]
    x = (rw['T'] - TC_BINDER) * n**(1/nu_collapse)
    y = rw['chi_conn'] / n**gamma_nu_peak
    y_err = err['chi_err'] / n**gamma_nu_peak
    mask = (x > -15) & (x < 15)
    c = colors_rw[i]
    ax.plot(x[mask], y[mask], 'o-', color=c, markersize=2, lw=0.6, label=f'$L={n}$')
    ax.fill_between(x[mask], (y - y_err)[mask], (y + y_err)[mask], color=c, alpha=0.1)

ax.set_xlabel('$(T - T_c) L^{1/\\nu}$')
ax.set_ylabel('$\\chi_{\\mathrm{conn}} / L^{\\gamma/\\nu}$')
ax.legend(fontsize=5, ncol=2)
ax.set_xlim(-15, 15)
label_panel(ax, 'a', x=-0.15)
plt.tight_layout()
plt.savefig('fss_collapse.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Summary Table

In [ ]:
results = {
    'Tc (Binder)':  (TC_BINDER,      TC_THEORY, abs(TC_BINDER - TC_THEORY)/TC_THEORY*100),
    'nu (Binder dU/dT)': (nu_binder,  NU,        abs(nu_binder - NU)/NU*100),
    'nu (chi collapse)':   (nu_collapse, NU,        abs(nu_collapse - NU)/NU*100),
    'gamma/nu (peak)':   (gamma_nu_peak,  GAMMA/NU,  abs(gamma_nu_peak - GAMMA/NU)/(GAMMA/NU)*100),
    'beta/nu (M@Tc)':   (beta_nu_meas,   BETA/NU,   abs(beta_nu_meas - BETA/NU)/(BETA/NU)*100),
    '2beta/nu + gamma/nu':   (hyperscaling,   3.0,       abs(hyperscaling - 3.0)/3.0*100),
}

print(f'{"Quantity":<20} {"Measured":>10} {"Theory":>10} {"Error":>8}  Status')
print('=' * 60)
for name, (meas, theory, err) in results.items():
    status = 'OK' if err < 5 else '~' if err < 15 else 'BAD'
    print(f'{name:<20} {meas:>10.4f} {theory:>10.4f} {err:>7.1f}%  {status}')

print(f'\nNote: Peak scaling uses reweighted data for sizes {rw_sz}')
print(f'Note: All exponents extracted from histogram-reweighted fine T grid')

## 6. Global Scaling Collapse â€” Simultaneous Tc, Î½, Î³/Î½ Fit

The basic collapse above fixes Tc to theory and only varies Î½. Here we do a proper
3-parameter global fit: optimize **Tc, Î½, and the exponent ratio** simultaneously.

**Method** (Houdayer & Hartmann 2004): For each observable O(T, N), the scaling hypothesis says:
  O(T, N) = N^(x/Î½) Â· f((T - Tc) Â· N^(1/Î½))

If the collapse is perfect, nearby points from different N values should agree.
We measure collapse quality by binning the rescaled x-axis and computing the
variance within each bin. The best (Tc, Î½, x/Î½) minimizes total variance.

In [ ]:
# Global 3-parameter collapse using reweighted fine-grid data
# Use adjacent-point cost (Kawashima & Ito 1993) which is more robust than binned variance
def collapse_cost_adj(params, obs_col):
    """Adjacent-point collapse cost: sort all rescaled points by x, sum (dy)^2 / (range)^2."""
    tc, nu, exp_ratio = params
    if nu <= 0.2 or nu > 1.5:
        return 1e10
    all_x, all_y, all_n = [], [], []
    for n in rw_sz:
        rw = rw_dfs[n]
        x = (rw['T'].values - tc) * n**(1.0/nu)
        y = rw[obs_col].values / n**exp_ratio
        # Only include points in a reasonable window
        mask = (x >= -15) & (x <= 15)
        if mask.sum() < 3:
            return 1e10
        all_x.extend(x[mask]); all_y.extend(y[mask]); all_n.extend([n]*mask.sum())
    if len(all_x) < 20:
        return 1e10
    all_x, all_y = np.array(all_x), np.array(all_y)
    order = np.argsort(all_x)
    ys = all_y[order]
    rng = ys.max() - ys.min()
    if rng < 1e-15:
        return 1e10
    return float(np.sum(np.diff(ys)**2)) / rng**2

print("=== chi_conn global collapse ===")
# Use tighter bounds informed by peak scaling results
result_chi = differential_evolution(
    collapse_cost_adj, [(4.45, 4.55), (0.5, 0.9), (1.5, 2.5)],
    args=('chi_conn',), seed=42, maxiter=1000, tol=1e-10, polish=True
)
tc_chi, nu_chi, gamma_nu_fit = result_chi.x
print(f'chi_conn: Tc={tc_chi:.4f}, nu={nu_chi:.4f}, gamma/nu={gamma_nu_fit:.4f}')
print(f'  (theory: Tc={TC_THEORY}, nu={NU:.4f}, gamma/nu={GAMMA/NU:.4f})')

print("\n=== Binder global collapse ===")
def binder_cost_adj(params):
    tc, nu = params
    if nu <= 0.2 or nu > 1.5: return 1e10
    all_x, all_y = [], []
    for n in rw_sz:
        rw = rw_dfs[n]
        x = (rw['T'].values - tc) * n**(1.0/nu)
        mask = (x >= -15) & (x <= 15)
        if mask.sum() < 3:
            return 1e10
        all_x.extend(x[mask]); all_y.extend(rw['U'].values[mask])
    all_x, all_y = np.array(all_x), np.array(all_y)
    order = np.argsort(all_x)
    ys = all_y[order]
    rng = ys.max() - ys.min()
    if rng < 1e-15:
        return 1e10
    return float(np.sum(np.diff(ys)**2)) / rng**2

result_binder = differential_evolution(
    binder_cost_adj, [(4.45, 4.55), (0.5, 0.9)],
    seed=42, maxiter=1000, tol=1e-10, polish=True
)
tc_binder_c, nu_binder_c = result_binder.x
print(f'Binder: Tc={tc_binder_c:.4f}, nu={nu_binder_c:.4f}')
print(f'  (theory: Tc={TC_THEORY}, nu={NU:.4f})')

print(f'\n=== Combined estimates ===')
tc_all = [TC_BINDER, tc_chi, tc_binder_c]
nu_all = [nu_collapse, nu_chi, nu_binder_c]
print(f'Tc = {np.mean(tc_all):.4f} +/- {np.std(tc_all):.4f}  (theory = {TC_THEORY})')
print(f'nu  = {np.mean(nu_all):.4f} +/- {np.std(nu_all):.4f}  (theory = {NU})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
colors_rw = cm.viridis(np.linspace(0.15, 0.85, len(rw_sz)))

# (a) Chi collapse with globally optimised params
ax = axes[0]
for i, n in enumerate(rw_sz):
    rw = rw_dfs[n]
    err = rw_errs[n]
    x = (rw['T'] - tc_chi) * n**(1.0/nu_chi)
    y = rw['chi_conn'] / n**gamma_nu_fit
    y_err = err['chi_err'] / n**gamma_nu_fit
    mask = (x > -12) & (x < 12)
    c = colors_rw[i]
    ax.plot(x[mask], y[mask], 'o-', color=c, markersize=2, lw=0.6, label=f'$L={n}$')
    ax.fill_between(x[mask], (y - y_err)[mask], (y + y_err)[mask], color=c, alpha=0.1)
ax.set_xlabel('$(T - T_c) L^{1/\\nu}$')
ax.set_ylabel('$\\chi L^{-\\gamma/\\nu}$')
ax.set_xlim(-12, 12); ax.legend(fontsize=5, ncol=2)
label_panel(ax, 'a')

# (b) Binder collapse
ax = axes[1]
for i, n in enumerate(rw_sz):
    rw = rw_dfs[n]
    err = rw_errs[n]
    x = (rw['T'] - tc_binder_c) * n**(1.0/nu_binder_c)
    mask = (x > -12) & (x < 12)
    c = colors_rw[i]
    ax.plot(x[mask], rw.loc[mask, 'U'], 'o-', color=c, markersize=2, lw=0.6, label=f'$L={n}$')
    ax.fill_between(x[mask],
                    rw.loc[mask, 'U'] - err.loc[mask, 'U_err'],
                    rw.loc[mask, 'U'] + err.loc[mask, 'U_err'],
                    color=c, alpha=0.1)
ax.set_xlabel('$(T - T_c) L^{1/\\nu}$')
ax.set_ylabel('$U$')
ax.set_xlim(-12, 12); ax.legend(fontsize=5, ncol=2)
label_panel(ax, 'b')

plt.tight_layout()
plt.savefig('fss_collapse_global.png', dpi=300, bbox_inches='tight')
plt.show()

# === FINAL SUMMARY ===
print('=' * 65)
print('  FSS RESULTS (Wolff + histogram reweighting)')
print('=' * 65)
final = {
    'Tc (Binder crossing)': (TC_BINDER, TC_THEORY),
    'nu (Binder slope)':     (nu_binder, NU),
    'nu (chi collapse)':     (nu_collapse, NU),
    'gamma/nu (peak)':       (gamma_nu_peak, GAMMA/NU),
    'beta/nu (M at Tc)':     (beta_nu_meas, BETA/NU),
    'gamma/nu (global)':     (gamma_nu_fit, GAMMA/NU),
    '2b/n + g/n':            (hyperscaling, 3.0),
}
print(f'{"Quantity":<24} {"Measured":>10} {"Theory":>10} {"Error":>8}')
print('-' * 55)
for name, (meas, theory) in final.items():
    err = abs(meas - theory) / theory * 100
    print(f'{name:<24} {meas:>10.4f} {theory:>10.4f} {err:>7.1f}%')
print('-' * 55)

## 7. Ferrenberg-Swendsen Histogram Reweighting

Single-histogram reweighting: given raw (E, M) samples at simulation temperature Tâ‚€,
estimate observables at any nearby temperature T via:

$$\langle O \rangle_T = \frac{\sum_i O_i \, e^{-(\beta - \beta_0) E_i^{total}}}{\sum_i e^{-(\beta - \beta_0) E_i^{total}}}$$

where $E_i^{total} = e_i \cdot N^3$ is the total energy of sample $i$.

This lets us interpolate observables to arbitrary temperatures between simulation points,
resolving the critical peak even for large N where the coarse grid misses it.

**Generate raw data:**
```bash
cargo run --release --bin fss -- --wolff --raw --sizes 16,24,32,40,48 \
    --tmin 4.40 --tmax 4.60 --steps 21 --warmup 5000 --samples 20000 --outdir analysis/data
```

In [ ]:
# Reweighted data already computed above (rw_dfs, rw_errs)
# This cell just displays the reweighting summary
print(f'Reweighted sizes: {sorted(rw_dfs.keys())}')
print(f'Fine T grid: {T_fine[0]:.2f} to {T_fine[-1]:.2f}, {len(T_fine)} points')
for n in sorted(rw_dfs.keys()):
    rw = rw_dfs[n]
    idx = rw['chi_conn'].idxmax()
    print(f'  N={n}: chi_peak={rw.loc[idx,"chi_conn"]:.1f} +/- {rw_errs[n].loc[idx,"chi_err"]:.1f} at T={rw.loc[idx,"T"]:.4f}')

In [ ]:
# Plot reweighted observables with error bands
fig, axes = plt.subplots(1, 3, figsize=(7, 2.8))
rw_colors = cm.viridis(np.linspace(0.15, 0.85, len(rw_dfs)))

ax = axes[0]
for (n, rw), c in zip(sorted(rw_dfs.items()), rw_colors):
    err = rw_errs[n]
    ax.plot(rw['T'], rw['chi_conn'], color=c, lw=0.8, label=f'$L={n}$')
    ax.fill_between(rw['T'], rw['chi_conn'] - err['chi_err'],
                    rw['chi_conn'] + err['chi_err'], color=c, alpha=0.15)
ax.axvline(TC_THEORY, color='red', ls='--', lw=0.6)
ax.set_xlabel('$T \\; [J/k_B]$'); ax.set_ylabel('$\\chi_{\\mathrm{conn}}$')
label_panel(ax, 'a'); ax.legend(fontsize=5, ncol=2)

ax = axes[1]
for (n, rw), c in zip(sorted(rw_dfs.items()), rw_colors):
    err = rw_errs[n]
    ax.plot(rw['T'], rw['Cv'], color=c, lw=0.8, label=f'$L={n}$')
    ax.fill_between(rw['T'], rw['Cv'] - err['cv_err'],
                    rw['Cv'] + err['cv_err'], color=c, alpha=0.15)
ax.axvline(TC_THEORY, color='red', ls='--', lw=0.6)
ax.set_xlabel('$T \\; [J/k_B]$'); ax.set_ylabel('$C_v$')
label_panel(ax, 'b')

ax = axes[2]
for (n, rw), c in zip(sorted(rw_dfs.items()), rw_colors):
    err = rw_errs[n]
    ax.plot(rw['T'], rw['U'], color=c, lw=0.8, label=f'$L={n}$')
    ax.fill_between(rw['T'], rw['U'] - err['U_err'],
                    rw['U'] + err['U_err'], color=c, alpha=0.15)
ax.axvline(TC_THEORY, color='red', ls='--', lw=0.6)
ax.set_xlabel('$T \\; [J/k_B]$'); ax.set_ylabel('Binder $U$')
label_panel(ax, 'c')

plt.tight_layout()
plt.savefig('fss_reweighted.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Reweighted peak scaling with error bars (same data, formatted for paper)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
Nfit = np.linspace(min(rw_sz)*0.9, max(rw_sz)*1.1, 100)

ax = axes[0]
ax.errorbar(rw_sz, chi_peaks, yerr=chi_peak_errs,
            fmt='o', ms=5, color=PUB_COLORS[0], capsize=3, zorder=5)
c0 = np.log(chi_peaks[0]) - slope_chi*np.log(rw_sz[0])
ax.plot(Nfit, np.exp(slope_chi*np.log(Nfit)+c0), '--', color=PUB_COLORS[1], lw=0.8,
        label=f'$\\gamma/\\nu={gamma_nu_peak:.3f}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$L$'); ax.set_ylabel('$\\chi_{\\mathrm{max}}$')
ax.legend(fontsize=6); label_panel(ax, 'a')

ax = axes[1]
ax.errorbar(rw_sz, m_at_tc, yerr=m_at_tc_errs,
            fmt='o', ms=5, color=PUB_COLORS[0], capsize=3, zorder=5)
c0m = np.log(m_at_tc[0]) - slope_m*np.log(rw_sz[0])
ax.plot(Nfit, np.exp(slope_m*np.log(Nfit)+c0m), '--', color=PUB_COLORS[1], lw=0.8,
        label=f'$\\beta/\\nu={beta_nu_meas:.3f}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$L$'); ax.set_ylabel('$M(T_c)$')
ax.legend(fontsize=6); label_panel(ax, 'b')

plt.tight_layout()
plt.savefig('fss_reweighted_scaling.png', dpi=300, bbox_inches='tight')
plt.show()